# Sistemas Inteligentes I — Baseline
## Búsqueda informada: Costo Uniforme, A* y Beam Search

Versión limpia del notebook original para usar como punto de partida.

- **Costo Uniforme:** prioriza el menor $g(n)$.
- **A*:** prioriza el menor $g(n)+h(n)$.
- **Beam Search:** conserva solo los mejores $k$ candidatos de cada nivel.
- **Heurística admisible:** no sobreestima el costo restante.

> La lógica de los ejercicios originales se conserva. Se eliminaron celdas repetitivas de depuración y texto que no es necesario para modificar el taller.


# 1. Idea central

En BFS y DFS la frontera se organiza sin estimar qué nodo está más cerca del objetivo. En búsqueda informada podemos incorporar información adicional.

- $g(n)$: costo acumulado desde el inicio hasta $n$.
- $h(n)$: estimación del costo desde $n$ hasta el objetivo.
- En A*: $f(n)=g(n)+h(n)$.

**Costo Uniforme:** prioriza menor $g(n)$.  
**A\*:** prioriza menor $g(n)+h(n)$.  
**Beam Search:** conserva solo los mejores $k$ candidatos de cada nivel.

In [1]:
from heapq import heappush, heappop
from itertools import count
import math
import matplotlib.pyplot as plt

print("Entorno listo")

Entorno listo


# 2. Problema base: grafo ponderado

- Estado inicial: `A`
- Objetivo: `G`
- Cada arista tiene un costo.

In [2]:
grafo = {
    "A": {"B": 2, "C": 4},
    "B": {"D": 5, "E": 10},
    "C": {"E": 3, "F": 6},
    "D": {"G": 4},
    "E": {"G": 2},
    "F": {"G": 3},
    "G": {}
}
grafo

{'A': {'B': 2, 'C': 4},
 'B': {'D': 5, 'E': 10},
 'C': {'E': 3, 'F': 6},
 'D': {'G': 4},
 'E': {'G': 2},
 'F': {'G': 3},
 'G': {}}

# 3. Búsqueda de Costo Uniforme (UCS)

Expande el nodo con menor costo acumulado:

$$f(n)=g(n)$$

No usa heurística. La frontera se implementa con una cola de prioridad.

In [3]:
def costo_uniforme(grafo, inicio, objetivo):
    orden = count()
    frontera = []
    heappush(frontera, (0, next(orden), inicio, [inicio]))
    mejor_g = {inicio: 0}
    expandidos = 0

    while frontera:
        g, _, nodo, camino = heappop(frontera)
        if g != mejor_g.get(nodo):
            continue
        expandidos += 1
        if nodo == objetivo:
            return {"camino": camino, "costo": g, "expandidos": expandidos}

        for vecino, costo in grafo[nodo].items():
            nuevo_g = g + costo
            if nuevo_g < mejor_g.get(vecino, math.inf):
                mejor_g[vecino] = nuevo_g
                heappush(frontera, (nuevo_g, next(orden), vecino, camino + [vecino]))
    return None

resultado_ucs = costo_uniforme(grafo, "A", "G")
resultado_ucs

{'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 6}

## 3.1 UCS paso a paso

In [4]:
def ucs_debug(grafo, inicio, objetivo):
    orden = count(); frontera=[]
    heappush(frontera,(0,next(orden),inicio,[inicio]))
    mejor_g={inicio:0}; paso=1
    while frontera:
        print(f"\nPaso {paso}")
        print("Frontera:", [(g,n) for g,_,n,_ in frontera])
        g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo):
            continue
        print(f"Expandimos {nodo}: g={g}")
        if nodo==objetivo: return camino,g
        for vecino,costo in grafo[nodo].items():
            ng=g+costo
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                heappush(frontera,(ng,next(orden),vecino,camino+[vecino]))
        paso += 1

ucs_debug(grafo,"A","G")


Paso 1
Frontera: [(0, 'A')]
Expandimos A: g=0

Paso 2
Frontera: [(2, 'B'), (4, 'C')]
Expandimos B: g=2

Paso 3
Frontera: [(4, 'C'), (7, 'D'), (12, 'E')]
Expandimos C: g=4

Paso 4
Frontera: [(7, 'D'), (10, 'F'), (7, 'E'), (12, 'E')]
Expandimos D: g=7

Paso 5
Frontera: [(7, 'E'), (10, 'F'), (12, 'E'), (11, 'G')]
Expandimos E: g=7

Paso 6
Frontera: [(9, 'G'), (10, 'F'), (12, 'E'), (11, 'G')]
Expandimos G: g=9


(['A', 'C', 'E', 'G'], 9)

# 4. Heurística

Hasta ahora, **Costo Uniforme** solo utiliza información sobre el costo ya recorrido:

\[
g(n)
\]

Para orientar la búsqueda hacia el objetivo podemos incorporar una estimación:

\[
h(n)
\]

donde \(h(n)\) representa una **estimación del costo mínimo restante** desde el nodo \(n\) hasta el objetivo.

## ¿De dónde salen estos valores?

En un problema real, una heurística debe derivarse de alguna propiedad del dominio: distancia geométrica, cantidad de elementos fuera de lugar, una relajación del problema, conocimiento experto, etc.

En este grafo pequeño conocemos toda la estructura, por lo que podemos calcular el costo real mínimo desde cada nodo hasta `G`. Lo usaremos únicamente con fines didácticos para evaluar la calidad de una heurística.

In [5]:
# Costo real mínimo desde cada nodo hasta G.
# En un problema real normalmente NO conoceríamos estos valores de antemano.

costo_real_hasta_G = {
    "A": 9,
    "B": 9,
    "C": 5,
    "D": 4,
    "E": 2,
    "F": 3,
    "G": 0,
}

costo_real_hasta_G

{'A': 9, 'B': 9, 'C': 5, 'D': 4, 'E': 2, 'F': 3, 'G': 0}

## 4.1 Costo real vs. estimación heurística

Denotaremos el costo real mínimo restante como:

$$
h^*(n)
$$

La heurística $$h(n)$$ intenta aproximarlo:

$$
h(n)\approx h^*(n)
$$

Una heurística es **admisible** cuando nunca sobreestima el costo real:

$$
\boxed{h(n)\leq h^*(n)}
$$

Para este ejemplo utilizaremos una heurística **admisible pero imperfecta**:

| Nodo | $$h(n)$$ | $$h^*(n)$$ |
|---|---:|---:|
| A | 7 | 9 |
| B | 6 | 9 |
| C | 4 | 5 |
| D | 3 | 4 |
| E | 2 | 2 |
| F | 2 | 3 |
| G | 0 | 0 |

La heurística no conoce exactamente la solución: solo proporciona una estimación razonable.

In [6]:
heuristica = {
    "A": 7,
    "B": 6,
    "C": 4,
    "D": 3,
    "E": 2,
    "F": 2,
    "G": 0,
}

heuristica

{'A': 7, 'B': 6, 'C': 4, 'D': 3, 'E': 2, 'F': 2, 'G': 0}

In [7]:
# Verificación de admisibilidad
comparacion_heuristica = {
    nodo: {
        "h(n)": heuristica[nodo],
        "h*(n)": costo_real_hasta_G[nodo],
        "admisible": heuristica[nodo] <= costo_real_hasta_G[nodo],
    }
    for nodo in heuristica
}

comparacion_heuristica

{'A': {'h(n)': 7, 'h*(n)': 9, 'admisible': True},
 'B': {'h(n)': 6, 'h*(n)': 9, 'admisible': True},
 'C': {'h(n)': 4, 'h*(n)': 5, 'admisible': True},
 'D': {'h(n)': 3, 'h*(n)': 4, 'admisible': True},
 'E': {'h(n)': 2, 'h*(n)': 2, 'admisible': True},
 'F': {'h(n)': 2, 'h*(n)': 3, 'admisible': True},
 'G': {'h(n)': 0, 'h*(n)': 0, 'admisible': True}}

### Idea clave

> **La heurística no tiene que acertar exactamente. Tiene que estimar.**

Una heurística demasiado débil puede aportar poca información.  
Una heurística que sobreestima puede orientar de manera agresiva la búsqueda, pero puede perder garantías de optimalidad.

# 5. A*

A* combina costo acumulado y estimación:

$$\boxed{f(n)=g(n)+h(n)}$$

Se expande primero el nodo con menor $f(n)$.

In [8]:
def a_estrella(grafo, heuristica, inicio, objetivo):
    tie = count(); frontera=[]
    heappush(frontera,(heuristica[inicio],0,next(tie),inicio,[inicio]))
    mejor_g={inicio:0}; expandidos=0

    while frontera:
        f,g,_,nodo,camino = heappop(frontera)
        if g != mejor_g.get(nodo):
            continue
        expandidos += 1
        if nodo == objetivo:
            return {"camino":camino,"costo":g,"expandidos":expandidos}

        for vecino,costo in grafo[nodo].items():
            ng=g+costo
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                nf=ng+heuristica[vecino]
                heappush(frontera,(nf,ng,next(tie),vecino,camino+[vecino]))
    return None

resultado_astar = a_estrella(grafo,heuristica,"A","G")
resultado_astar

{'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 5}

## 5.1 A* paso a paso

In [9]:
def astar_debug(grafo, heuristica, inicio, objetivo):
    tie=count(); frontera=[]
    heappush(frontera,(heuristica[inicio],0,next(tie),inicio,[inicio]))
    mejor_g={inicio:0}; paso=1
    while frontera:
        print(f"\nPaso {paso}")
        print("Frontera:", [(n,g,f) for f,g,_,n,_ in frontera])
        f,g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo): continue
        print(f"Expandimos {nodo}: g={g}, h={heuristica[nodo]}, f={f}")
        if nodo==objetivo: return camino,g
        for vecino,costo in grafo[nodo].items():
            ng=g+costo
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                nf=ng+heuristica[vecino]
                heappush(frontera,(nf,ng,next(tie),vecino,camino+[vecino]))
        paso += 1

astar_debug(grafo,heuristica,"A","G")


Paso 1
Frontera: [('A', 0, 7)]
Expandimos A: g=0, h=7, f=7

Paso 2
Frontera: [('B', 2, 8), ('C', 4, 8)]
Expandimos B: g=2, h=6, f=8

Paso 3
Frontera: [('C', 4, 8), ('D', 7, 10), ('E', 12, 14)]
Expandimos C: g=4, h=4, f=8

Paso 4
Frontera: [('E', 7, 9), ('F', 10, 12), ('D', 7, 10), ('E', 12, 14)]
Expandimos E: g=7, h=2, f=9

Paso 5
Frontera: [('G', 9, 9), ('D', 7, 10), ('E', 12, 14), ('F', 10, 12)]
Expandimos G: g=9, h=0, f=9


(['A', 'C', 'E', 'G'], 9)

## 5.2 Tres casos para entender el papel de la heurística

Antes de comparar los algoritmos, observemos qué ocurre con A* bajo tres configuraciones distintas de \(h(n)\).

### Caso 1 — Sin heurística

Si:

$$
h(n)=0 \quad \forall n
$$

entonces:

$$
f(n)=g(n)
$$

y A* se comporta como **Costo Uniforme**.

In [10]:
heuristica_cero = {nodo: 0 for nodo in grafo}

resultado_astar_h0 = a_estrella(
    grafo,
    heuristica_cero,
    "A",
    "G"
)

resultado_astar_h0

{'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 6}

### Caso 2 — Heurística admisible

Usamos la heurística definida anteriormente:

$$
h(n)\leq h^*(n)
$$

La búsqueda dispone de información para orientar la expansión sin sobreestimar el costo real restante.

In [11]:
resultado_astar_admisible = a_estrella(
    grafo,
    heuristica,
    "A",
    "G"
)

resultado_astar_admisible

{'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 5}

### Caso 3 — Heurística que sobreestima

Ahora construiremos deliberadamente una heurística **no admisible**.

Algunos valores cumplen:

$$
h(n)>h^*(n)
$$

Esto no significa que A* necesariamente encuentre una mala solución, pero ya no podemos asumir las mismas garantías de optimalidad.

In [12]:
heuristica_sobreestima = {
    "A": 12,  # costo real = 9
    "B": 10,  # costo real = 9
    "C": 4,
    "D": 3,
    "E": 7,   # costo real = 2
    "F": 2,
    "G": 0,
}

comparacion_sobreestimacion = {
    nodo: {
        "h(n)": heuristica_sobreestima[nodo],
        "h*(n)": costo_real_hasta_G[nodo],
        "sobreestima": heuristica_sobreestima[nodo] > costo_real_hasta_G[nodo],
    }
    for nodo in heuristica_sobreestima
}

comparacion_sobreestimacion

{'A': {'h(n)': 12, 'h*(n)': 9, 'sobreestima': True},
 'B': {'h(n)': 10, 'h*(n)': 9, 'sobreestima': True},
 'C': {'h(n)': 4, 'h*(n)': 5, 'sobreestima': False},
 'D': {'h(n)': 3, 'h*(n)': 4, 'sobreestima': False},
 'E': {'h(n)': 7, 'h*(n)': 2, 'sobreestima': True},
 'F': {'h(n)': 2, 'h*(n)': 3, 'sobreestima': False},
 'G': {'h(n)': 0, 'h*(n)': 0, 'sobreestima': False}}

In [13]:
resultado_astar_sobreestima = a_estrella(
    grafo,
    heuristica_sobreestima,
    "A",
    "G"
)

resultado_astar_sobreestima

{'camino': ['A', 'B', 'D', 'G'], 'costo': 11, 'expandidos': 5}

### Comparación de los tres casos

Compare:

- camino encontrado,
- costo total,
- nodos expandidos.

La idea importante es que una heurística no admisible **no necesariamente falla siempre**; simplemente deja de cumplir las condiciones que permiten garantizar la optimalidad de A*.

In [14]:
comparacion_tres_heuristicas = {
    "h(n)=0": resultado_astar_h0,
    "Admisible": resultado_astar_admisible,
    "Sobreestima": resultado_astar_sobreestima,
}

comparacion_tres_heuristicas

{'h(n)=0': {'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 6},
 'Admisible': {'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 5},
 'Sobreestima': {'camino': ['A', 'B', 'D', 'G'], 'costo': 11, 'expandidos': 5}}

# 6. Comparación inicial

**Pregunta clave:** ¿qué ocurre si $h(n)=0$ para todos los nodos?  
A* se comporta como Costo Uniforme.

In [15]:
{
    "Costo Uniforme": resultado_ucs,
    "A*": resultado_astar
}

{'Costo Uniforme': {'camino': ['A', 'C', 'E', 'G'],
  'costo': 9,
  'expandidos': 6},
 'A*': {'camino': ['A', 'C', 'E', 'G'], 'costo': 9, 'expandidos': 5}}

# 7. Caso aplicado: cuadrícula

Ahora trabajaremos con un mapa:

- `0`: celda libre
- `1`: obstáculo
- acciones: arriba, abajo, izquierda y derecha
- costo de cada movimiento: 1

Para A* usaremos distancia Manhattan:

$$h(n)=|x_n-x_g|+|y_n-y_g|$$

In [16]:
mapa = [
    [0,0,0,0,0,0,0],
    [0,1,1,1,0,1,0],
    [0,0,0,1,0,1,0],
    [1,1,0,1,0,0,0],
    [0,0,0,0,1,1,0],
    [0,1,1,0,0,0,0],
    [0,0,0,0,1,0,0],
]
INICIO=(0,0); OBJETIVO=(6,6)
MOVIMIENTOS=[(1,0),(-1,0),(0,1),(0,-1)]

def sucesores_mapa(mapa,pos):
    filas=len(mapa); cols=len(mapa[0]); f,c=pos; out=[]
    for df,dc in MOVIMIENTOS:
        nf,nc=f+df,c+dc
        if 0<=nf<filas and 0<=nc<cols and mapa[nf][nc]==0:
            out.append((nf,nc))
    return out

def manhattan(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])

## 7.1 Costo Uniforme en la cuadrícula

In [17]:
def ucs_mapa(mapa,inicio,objetivo):
    tie=count(); frontera=[(0,next(tie),inicio,[inicio])]
    mejor_g={inicio:0}; expandidos=0
    while frontera:
        g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo): continue
        expandidos += 1
        if nodo==objetivo:
            return {"camino":camino,"costo":g,"expandidos":expandidos}
        for vecino in sucesores_mapa(mapa,nodo):
            ng=g+1
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                heappush(frontera,(ng,next(tie),vecino,camino+[vecino]))
    return None

resultado_ucs_mapa=ucs_mapa(mapa,INICIO,OBJETIVO)
resultado_ucs_mapa

{'camino': [(0, 0),
  (1, 0),
  (2, 0),
  (2, 1),
  (2, 2),
  (3, 2),
  (4, 2),
  (4, 3),
  (5, 3),
  (5, 4),
  (5, 5),
  (6, 5),
  (6, 6)],
 'costo': 12,
 'expandidos': 35}

## 7.2 A* en la cuadrícula

In [18]:
def astar_mapa(mapa,inicio,objetivo):
    tie=count(); frontera=[(manhattan(inicio,objetivo),0,next(tie),inicio,[inicio])]
    mejor_g={inicio:0}; expandidos=0
    while frontera:
        f,g,_,nodo,camino=heappop(frontera)
        if g != mejor_g.get(nodo): continue
        expandidos += 1
        if nodo==objetivo:
            return {"camino":camino,"costo":g,"expandidos":expandidos}
        for vecino in sucesores_mapa(mapa,nodo):
            ng=g+1
            if ng < mejor_g.get(vecino, math.inf):
                mejor_g[vecino]=ng
                nf=ng+manhattan(vecino,objetivo)
                heappush(frontera,(nf,ng,next(tie),vecino,camino+[vecino]))
    return None

resultado_astar_mapa=astar_mapa(mapa,INICIO,OBJETIVO)
resultado_astar_mapa

{'camino': [(0, 0),
  (1, 0),
  (2, 0),
  (2, 1),
  (2, 2),
  (3, 2),
  (4, 2),
  (4, 3),
  (5, 3),
  (5, 4),
  (5, 5),
  (6, 5),
  (6, 6)],
 'costo': 12,
 'expandidos': 29}

## 7.3 Visualización

In [19]:
def mostrar_mapa(mapa, camino=None, inicio=None, objetivo=None, titulo="Mapa"):
    fig,ax=plt.subplots(figsize=(7,7))
    ax.imshow(mapa,cmap='cool')
    if camino:
        xs=[c for f,c in camino]; ys=[f for f,c in camino]
        ax.plot(xs,ys,c='red',marker="o")
    if inicio: ax.text(inicio[1],inicio[0],"Inicio",ha="center",va="center",fontsize=14)
    if objetivo: ax.text(objetivo[1],objetivo[0],"Final",ha="center",va="center",fontsize=14)
    ax.set_xticks(range(len(mapa[0]))); ax.set_yticks(range(len(mapa)))
    ax.set_title(titulo); ax.grid(True); plt.show()

mostrar_mapa(mapa,resultado_ucs_mapa["camino"],INICIO,OBJETIVO,"Costo Uniforme")
mostrar_mapa(mapa,resultado_astar_mapa["camino"],INICIO,OBJETIVO,"A* con Manhattan")

/tmp/ipykernel_401261/1948330058.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title(titulo); ax.grid(True); plt.show()


## 7.4 Comparación experimental

In [20]:
{
    "Costo Uniforme": {"costo":resultado_ucs_mapa["costo"],"expandidos":resultado_ucs_mapa["expandidos"]},
    "A*": {"costo":resultado_astar_mapa["costo"],"expandidos":resultado_astar_mapa["expandidos"]}
}

{'Costo Uniforme': {'costo': 12, 'expandidos': 35},
 'A*': {'costo': 12, 'expandidos': 29}}

# 8. Beam Search

Beam Search limita deliberadamente la cantidad de estados conservados en cada nivel.

$$k=\text{beam width}$$

Si $k=2$, solo se mantienen los dos estados más prometedores del nivel.

Ventaja: reduce memoria y exploración.  
Riesgo: puede descartar un estado que conduzca a la mejor solución. Por eso no garantiza optimalidad y puede fallar aunque exista solución.

In [21]:
def beam_search_mapa(mapa,inicio,objetivo,beam_width=2):
    haz=[(inicio,[inicio])]
    visitados={inicio}
    expandidos=0

    while haz:
        candidatos=[]
        for nodo,camino in haz:
            expandidos += 1
            if nodo==objetivo:
                return {"camino":camino,"costo":len(camino)-1,"expandidos":expandidos,"beam_width":beam_width}
            for vecino in sucesores_mapa(mapa,nodo):
                if vecino not in visitados:
                    visitados.add(vecino)
                    candidatos.append((vecino,camino+[vecino]))

        candidatos.sort(key=lambda x: manhattan(x[0],objetivo))
        haz=candidatos[:beam_width]
    return None

resultado_beam=beam_search_mapa(mapa,INICIO,OBJETIVO,beam_width=3)
resultado_beam

{'camino': [(0, 0),
  (1, 0),
  (2, 0),
  (2, 1),
  (2, 2),
  (3, 2),
  (4, 2),
  (4, 3),
  (5, 3),
  (5, 4),
  (5, 5),
  (6, 5),
  (6, 6)],
 'costo': 12,
 'expandidos': 31,
 'beam_width': 3}

## 8.1 Experimentar con diferentes anchos de haz

In [22]:
resultados_beam={}
for k in [1,2,3,5,10]:
    resultados_beam[k]=beam_search_mapa(mapa,INICIO,OBJETIVO,beam_width=k)
resultados_beam

{1: None,
 2: {'camino': [(0, 0),
   (1, 0),
   (2, 0),
   (2, 1),
   (2, 2),
   (3, 2),
   (4, 2),
   (4, 3),
   (5, 3),
   (5, 4),
   (5, 5),
   (6, 5),
   (6, 6)],
  'costo': 12,
  'expandidos': 24,
  'beam_width': 2},
 3: {'camino': [(0, 0),
   (1, 0),
   (2, 0),
   (2, 1),
   (2, 2),
   (3, 2),
   (4, 2),
   (4, 3),
   (5, 3),
   (5, 4),
   (5, 5),
   (6, 5),
   (6, 6)],
  'costo': 12,
  'expandidos': 31,
  'beam_width': 3},
 5: {'camino': [(0, 0),
   (1, 0),
   (2, 0),
   (2, 1),
   (2, 2),
   (3, 2),
   (4, 2),
   (4, 3),
   (5, 3),
   (5, 4),
   (5, 5),
   (6, 5),
   (6, 6)],
  'costo': 12,
  'expandidos': 35,
  'beam_width': 5},
 10: {'camino': [(0, 0),
   (1, 0),
   (2, 0),
   (2, 1),
   (2, 2),
   (3, 2),
   (4, 2),
   (4, 3),
   (5, 3),
   (5, 4),
   (5, 5),
   (6, 5),
   (6, 6)],
  'costo': 12,
  'expandidos': 35,
  'beam_width': 10}}

# 9. Comparación global

In [23]:
{
    "Costo Uniforme": {"costo":resultado_ucs_mapa["costo"],"expandidos":resultado_ucs_mapa["expandidos"]},
    "A*": {"costo":resultado_astar_mapa["costo"],"expandidos":resultado_astar_mapa["expandidos"]},
    "Beam Search k=3": None if resultado_beam is None else {"costo":resultado_beam["costo"],"expandidos":resultado_beam["expandidos"]}
}

{'Costo Uniforme': {'costo': 12, 'expandidos': 35},
 'A*': {'costo': 12, 'expandidos': 29},
 'Beam Search k=3': {'costo': 12, 'expandidos': 31}}

## 9.1 Resumen conceptual

| Algoritmo | Prioriza | Heurística | ¿Óptimo? | Memoria |
|---|---|---:|---|---|
| Costo Uniforme | menor $g(n)$ | No | Sí, con costos no negativos | Puede ser alta |
| A* | menor $g(n)+h(n)$ | Sí | Sí, bajo condiciones adecuadas | Puede ser alta |
| Beam Search | mejores $k$ candidatos | Sí | No garantizado | Controlada |

**Idea clave:**
- Costo Uniforme sabe cuánto ha costado llegar.
- A* sabe cuánto ha costado llegar y estima cuánto falta.
- Beam Search sacrifica exhaustividad para controlar el tamaño de la búsqueda.

# 10. Interpretación de la calidad de una heurística

Ya observamos tres escenarios:

- **Heurística nula:** no aporta información y A* se reduce a Costo Uniforme.
- **Heurística admisible:** orienta la búsqueda sin sobreestimar el costo real restante.
- **Heurística no admisible:** puede priorizar de forma agresiva ciertos caminos, pero pierde la garantía de optimalidad.

### Para discutir

1. ¿Una heurística con valores más grandes es necesariamente mejor?
2. ¿Qué diferencia existe entre una heurística **informativa** y una heurística que simplemente sobreestima?
3. ¿Por qué no podemos usar \(h^*(n)\) directamente como heurística en un problema real?
4. ¿Qué ocurriría si calcular \(h(n)\) fuera casi tan costoso como resolver el problema?
5. ¿Qué propiedades del dominio podrían utilizarse para construir una buena heurística?

### Respuestas

1. No. En el grafo de clase la heurística que sobreestima usó valores más grandes (A vale 12 frente a un costo real de 9, y E vale 7 frente a 2) y expandió 5 nodos, igual que la admisible, pero devolvió costo 11 en lugar de 9. Un número más alto no orienta mejor si se pasa de $h^*(n)$.

2. Una heurística informativa se acerca al costo restante sin pasarse. La admisible (7, 6, 4, …, todas menores o iguales que $h^*(n)$) conservó el camino A-C-E-G de costo 9 y bajó de 6 a 5 expansiones. La que sobreestima infló $f(n)$ en E y A* prefirió A-B-D-G, que cuesta 11.

3. $h^*(n)$ es el costo real que falta. Para conocerlo habría que haber resuelto el problema desde cada nodo. En este notebook aparece la tabla costo_real_hasta_G porque el grafo es pequeño. En un mapa real esa tabla no existe de antemano.

4. El tiempo que se ahorra al expandir menos nodos se gasta calculando $h(n)$. A* dejaría de ser más barato que una búsqueda más simple. Por eso sirve Manhattan: se calcula rápido y todavía informa.

5. Distancias que ignoran obstáculos, fichas fuera de lugar, simetrías o una versión relajada de los operadores. En la cuadrícula usé Manhattan. En el 8-puzzle comparé fichas fuera de lugar con Manhattan.


# 11. Reto: A* para el 8-puzzle

Compare tres estrategias sobre el mismo problema:

1. BFS
2. A* con fichas fuera de lugar
3. A* con distancia Manhattan

Mida:
- longitud de la solución;
- estados expandidos;
- tiempo de ejecución.

Heurísticas sugeridas:

$$h_1(n)=\text{número de fichas fuera de lugar}$$

$$h_2(n)=\sum_i (|x_i-x_i^*|+|y_i-y_i^*|)$$

### La implementación de A* para el 8-puzzle se encuentra en el [Notebook](3-astar-8puzzle.ipynb) en esta misma carpeta, con sus [resultados](../../results/informed-search/3-astar-8puzzle.md).


# 12. Preguntas de cierre

1. ¿Cuál es la diferencia entre $g(n)$ y $h(n)$?
2. ¿Por qué Costo Uniforme no es una búsqueda heurística?
3. ¿Por qué A* puede expandir menos nodos que Costo Uniforme?
4. ¿Qué pierde Beam Search al limitar la frontera?
5. ¿Qué algoritmo elegiría si necesita garantizar la solución de menor costo?
6. ¿Cuál elegiría si dispone de memoria muy limitada?
7. ¿Qué papel cumple la calidad de la heurística?

### Respuestas

1. $g(n)$ es el costo ya pagado desde el inicio. $h(n)$ estima lo que falta hasta la meta. En A* se suman: $f(n)=g(n)+h(n)$.

2. Costo Uniforme solo mira $g(n)$. En el grafo expandió 6 nodos y halló costo 9, igual que A* con $h(n)=0$. No consulta una heurística.

3. Porque $h(n)$ ordena la frontera hacia la meta. En el grafo A* expandió 5 nodos contra 6 de Costo Uniforme, con el mismo camino A-C-E-G de costo 9. En la cuadrícula de clase, 29 contra 35 y costo 12. En el mapa 12 × 12 del taller, 36 contra 49 y costo 22.

4. Descarta candidatos. Con $k=1$ en el mapa de clase no devolvió solución. En el taller 12 × 12, $k=1$ devolvió costo 40 en lugar de 22.

5. Costo Uniforme, o A* con una heurística admisible. Los dos conservaron costo 9 en el grafo y 22 en el mapa del taller.

6. Beam Search, porque la frontera queda acotada por $k$. El costo de esa decisión se vio con $k=1$.

7. Decide cuántos nodos extra se expanden sin perder el óptimo. Manhattan orientó mejor que la euclidiana (36 contra 37 expansiones) y mucho mejor que fichas fuera de lugar en el 8-puzzle (21198 contra 143849).


# 13. Taller posterior

### Parte 1 — Cuadrícula
Construya un mapa de al menos 12 × 12 y ejecute:
- Costo Uniforme;
- A*;
- Beam Search con $k = 1, 2, 4, 8$.

Compare costo de solución, estados expandidos y capacidad para encontrar solución.

Ver en [archivo](1-grid-search.ipynb) con sus [resultados](../../results/informed-search/1-grid-search.md)

### Parte 2 — Heurísticas
Implemente dos heurísticas distintas para A* y analice cuál orienta mejor la búsqueda.

Ver en [archivo](2-heuristics.ipynb) con sus [resultados](../../results/informed-search/2-heuristics.md)

### Parte 3 — 8-puzzle
Implemente A* con fichas fuera de lugar y distancia Manhattan y compare con BFS.

Ver en [archivo](3-astar-8puzzle.ipynb) con sus [resultados](../../results/informed-search/3-astar-8puzzle.md)

### Pregunta final
**¿Puede una búsqueda más rápida producir una solución peor? Explique usando sus experimentos.**

Ver en [archivo](../../results/informed-search/4-faster-worse.md)

### Respuesta

Sí. En el mapa de 12 × 12, Beam Search con k = 1 expandió 41 estados y devolvió un camino de costo 40. Costo Uniforme expandió 49 y A* 36, y los dos devolvieron costo 22. El haz corto recorre menos, pero se queda en el corredor que se ve cerca del objetivo y pierde el rodeo óptimo.

A* con Manhattan también recorre menos que Costo Uniforme, 36 contra 49, y conserva el mismo costo. Ahí la búsqueda más dirigida no empeora la solución. La diferencia está en qué se recorta: una heurística admisible ordena la frontera; un haz estrecho descarta candidatos y puede dejar afuera el camino de menor costo.

### Uso de IA generativa

- herramienta utilizada: Cursor
- propósito de uso: organizar el baseline, redactar respuestas y enlazar los talleres
- partes de la solución en las que fue empleada: este notebook y los talleres de la misma carpeta


# 14. Conclusiones y transición

- Costo Uniforme prioriza el costo acumulado.
- A* combina costo acumulado y estimación del costo restante.
- Una buena heurística puede reducir la exploración.
- Beam Search reduce el uso de recursos, pero sacrifica garantías.
